## 1)What is the Expectation of a Random Variable?

The **expectation** (also called the **expected value** or **mean**) of a random variable is the "average value" you would get if you repeated an experiment a very large number of times.

Think of it as the center of gravity of a distribution. It tells you where the distribution is "balanced".

### A Simple Example

Roll a fair six-sided die. The possible outcomes are 1, 2, 3, 4, 5, 6, each with probability 1/6.

The expected value is:

$$E[X] = 1 \cdot \frac{1}{6} + 2 \cdot \frac{1}{6} + 3 \cdot \frac{1}{6} + 4 \cdot \frac{1}{6} + 5 \cdot \frac{1}{6} + 6 \cdot \frac{1}{6} = 3.5$$

You will never actually roll a 3.5, but if you rolled the die thousands of times and averaged the results, you would get very close to 3.5.

### The General Formula

For a **discrete** random variable X with possible values $x_i$ and probabilities $p(x_i)$:

$$E[X] = \sum_{i} x_i \cdot p(x_i)$$

For a **continuous** random variable X with probability density function $p(x)$:

$$E[X] = \int x \cdot p(x) \, dx$$

Both say the same thing: sum up each possible value, weighted by how likely it is.

### Expectation of a Function

You can also take the expectation of a function of X, written $E[f(X)]$:

$$E[f(X)] = \sum_{i} f(x_i) \cdot p(x_i) \quad \text{(discrete)}$$

$$E[f(X)] = \int f(x) \cdot p(x) \, dx \quad \text{(continuous)}$$

This comes up constantly in machine learning. For example, the loss function is often written as an expectation over the data distribution:

$$\mathcal{L} = E_{x \sim p(x)}[\text{loss}(x)]$$

This just means: "the average loss over all data points, weighted by how often each point appears."

### Key Properties

| Property | Formula | What it means |
|---|---|---|
| Linearity | $E[aX + b] = aE[X] + b$ | Scaling and shifting pass through |
| Sum rule | $E[X + Y] = E[X] + E[Y]$ | Always true, even if X and Y are dependent |
| Independence | $E[XY] = E[X] \cdot E[Y]$ | Only true when X and Y are independent |

> **Why this matters for diffusion models:** The ELBO and the training objectives for VAEs and diffusion models are all written as expectations. Understanding expectation is the foundation for reading and deriving those objectives.

## 2) Self-Information (Surprise of an Outcome)

**Self-information** tells you how "surprising" a single outcome is.

The core idea is simple: rare events are more surprising than common ones.

- If someone says "the sun rose this morning" you are not surprised. That event has very high probability, so it carries almost no information.
- If someone says "it snowed in the Sahara today" you are very surprised. That event has very low probability, so it carries a lot of information.

### The Formula

For an outcome $x$ with probability $p(x)$:

$$I(x) = -\log p(x) = \log \frac{1}{p(x)}$$

The negative log turns the relationship around correctly:
- High probability ($p(x)$ close to 1) gives low surprise ($I(x)$ close to 0)
- Low probability ($p(x)$ close to 0) gives high surprise ($I(x)$ large)

### Example

| Event | Probability | Surprise $I(x)$ |
|---|---|---|
| Fair coin lands heads | 0.5 | $-\log_2(0.5) = 1$ bit |
| Rolling a 6 on a die | 1/6 | $-\log_2(1/6) \approx 2.58$ bits |
| Drawing the ace of spades | 1/52 | $-\log_2(1/52) \approx 5.70$ bits |

> The unit depends on the base of the log. Base 2 gives **bits**, base $e$ gives **nats**. In machine learning, nats (natural log) are almost always used.

## Entropy (Average Surprise)

**Entropy** is the expected surprise of a distribution. It answers:

> "On average, how surprised will I be by an outcome drawn from this distribution?"

It is just the expectation of self-information:

$$H(p) = E_{x \sim p}[I(x)] = E_{x \sim p}[-\log p(x)] = -\sum_{x} p(x) \log p(x)$$

### Intuition

- A distribution that is very **peaked** (one outcome is almost certain) has **low entropy**. You are rarely surprised.
- A distribution that is very **spread out** (all outcomes are equally likely) has **high entropy**. Every outcome surprises you equally.

### Example: A Biased Coin

| Coin | P(heads) | P(tails) | Entropy |
|---|---|---|---|
| Always heads | 1.0 | 0.0 | 0 bits (no surprise ever) |
| Slightly biased | 0.9 | 0.1 | 0.47 bits |
| Fair coin | 0.5 | 0.5 | 1 bit (maximum surprise) |

A fair coin has maximum entropy because you have no idea what is coming next. A coin that always lands heads has zero entropy because there is nothing to be surprised about.

### The Formula (continuous)

$$H(p) = -\int p(x) \log p(x) \, dx$$

> **Key takeaway:** Entropy measures the irreducible uncertainty in a distribution. It is the baseline average surprise you must accept when you know the true distribution perfectly.

## Cross-Entropy (Average Surprise Under a Wrong Belief)

**Cross-entropy** asks: what is the average surprise if the true distribution is **p**, but you are using a different distribution **q** to measure surprise?

You are drawing events from reality (p), but using the wrong model (q) to judge how surprising they are.

$$H(p, q) = E_{x \sim p}[-\log q(x)] = -\sum_{x} p(x) \log q(x)$$

### Intuition

Imagine a weather forecaster (q) who believes every day has a 50/50 chance of rain. But reality (p) is a dry desert where it only rains 5% of the time.

When it does not rain (which happens 95% of the time), the forecaster thinks the probability was 0.5, so assigns surprise $-\log(0.5) = 1$ bit. But the true surprise should be much lower since rain is rare.

The forecaster is systematically over-surprised because their model q does not match reality p. This extra, unnecessary surprise is what cross-entropy captures.

### Cross-Entropy vs Entropy

| | Formula | What it means |
|---|---|---|
| Entropy $H(p)$ | $-\sum_x p(x) \log p(x)$ | Average surprise using the **correct** model |
| Cross-entropy $H(p, q)$ | $-\sum_x p(x) \log q(x)$ | Average surprise using the **wrong** model q |

Cross-entropy is always at least as large as entropy:

$$H(p, q) \geq H(p)$$

The gap between them is the extra surprise caused by using the wrong model. That gap has a name.

> **Spoiler:** That gap is exactly KL divergence.

## KL Divergence (Derived from Cross-Entropy and Entropy)

We now have everything we need to derive KL divergence from scratch.

### The Derivation

Recall:
- Entropy: $H(p) = -\sum_x p(x) \log p(x)$ (average surprise under the correct model)
- Cross-entropy: $H(p, q) = -\sum_x p(x) \log q(x)$ (average surprise under the wrong model)

KL divergence is defined as the **extra** average surprise you suffer by using q instead of p:

$$D_{KL}(p \| q) = H(p, q) - H(p)$$

Expanding:

$$D_{KL}(p \| q) = -\sum_x p(x) \log q(x) - \left(-\sum_x p(x) \log p(x)\right)$$

$$= \sum_x p(x) \log p(x) - \sum_x p(x) \log q(x)$$

$$= \sum_x p(x) \left(\log p(x) - \log q(x)\right)$$

$$\boxed{D_{KL}(p \| q) = \sum_x p(x) \log \frac{p(x)}{q(x)}}$$

For continuous distributions:

$$D_{KL}(p \| q) = \int p(x) \log \frac{p(x)}{q(x)} \, dx$$

It is also the expectation of log-ratio under p:

$$D_{KL}(p \| q) = E_{x \sim p}\left[\log \frac{p(x)}{q(x)}\right]$$

### The Full Picture

$$\underbrace{H(p,q)}_{\text{cross-entropy}} = \underbrace{H(p)}_{\text{entropy}} + \underbrace{D_{KL}(p \| q)}_{\text{KL divergence}}$$

| Quantity | Measures |
|---|---|
| $H(p)$ | Minimum average surprise (irreducible, set by reality) |
| $D_{KL}(p \| q)$ | Extra surprise from using the wrong model q |
| $H(p, q)$ | Total average surprise when using q to model p |

### Key Properties

| Property | Detail |
|---|---|
| Always non-negative | $D_{KL}(p \| q) \geq 0$ always (since cross-entropy $\geq$ entropy) |
| Zero when identical | $D_{KL}(p \| q) = 0$ if and only if $p = q$ everywhere |
| Not symmetric | $D_{KL}(p \| q) \neq D_{KL}(q \| p)$ in general |

Because it is not symmetric, it is **not** a true distance metric. The order matters.

### Why This Matters for VAEs and Diffusion Models

In VAEs, the goal is to make the encoder output $q(z|x)$ as close as possible to the prior $p(z)$. This is enforced by adding a KL divergence term to the loss:

$$\mathcal{L} = \underbrace{E[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{regularisation}}$$

The KL term penalises the encoder for drifting away from the prior. Minimising it is equivalent to minimising the extra surprise that comes from the encoder using the wrong belief about z.

> **Key takeaway:** KL divergence is not an arbitrary formula. It is the natural answer to the question "how much worse is my model than the truth?" built directly from self-information, entropy, and cross-entropy.